# Module 3 - Prévision des ventes

Comparer Prophet à des baselines simples avant de l'utiliser pour le target-setting.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.sales_forecast import (
    compare_backtest,
    mape_summary,
    monthly_sales,
    prophet_forecast,
)
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_normalized.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'generated'

## 1. Préparer l'historique

La Pologne est exclue car elle ne contient qu'une seule année complète.

In [ ]:
transactions = pd.read_csv(INPUT_PATH, parse_dates=['Date'])
history = monthly_sales(transactions, country='Germany')
history.head()

## 2. Backtest 2017-2019 contre 2020

On compare la moyenne historique, le naïf saisonnier à 12 mois et Prophet.

In [ ]:
train = history[history['ds'] < '2020-01-01']
test = history[history['ds'] >= '2020-01-01']
backtest = compare_backtest(train, test)
metrics = mape_summary(backtest)
metrics.round(1)

## 3. Prévision exploratoire

Les intervalles Prophet sont conservés pour exploration. La baseline la plus performante reste la référence de décision.

In [ ]:
forecast = prophet_forecast(history, periods=12)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
history.to_csv(OUTPUT_DIR / 'monthly_sales_history.csv', index=False)
backtest.to_csv(OUTPUT_DIR / 'sales_forecast_backtest.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'sales_forecast_metrics.csv', index=False)
forecast.to_csv(OUTPUT_DIR / 'sales_forecast_prophet.csv', index=False)
forecast.tail()